Download all the requirements

In [ ]:
%pip install torch==2.7.1 torchvision torchaudio==2.7.1+cu126 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%pip install torchcodec ffmpeg-python pandas tqdm scikit-learn librosa birdnetlib birdnet resampy

In [ ]:
%pip freeze > requirements.txt

In [ ]:
# Download dataset
import kagglehub

# Download latest version
kagglehub.auth.set_kaggle_api_token('KGAT_fb4d6921c565524358c1914efc082ed4')
path = kagglehub.competition_download('birdclef-2026', output_dir=os.path.join("birdclef-2026"))

print("Path to competition files:", path)

Preprocessing the data

In [ ]:
# Helper - Get audio chunks with thresholding
from scipy import signal as scipy_signal
import numpy as np

# ──────────────────────────────────────────────────────────────────────────────
# Non-Aves: spectrogram peak detection via band-limited RMS
# ──────────────────────────────────────────────────────────────────────────────

def _get_spectrogram_active_chunks(
    audio_path:      str,
    chunk_sec:       float = 3.0,
    freq_min:        float = 100.0,
    freq_max:        float = 16_000.0,
    db_above_median: float = 6.0,
) -> list[tuple[float, float]]:
    """
    Detect active chunks via band-limited RMS energy.

    Reads the file once, band-pass filters to the species' vocal range, then
    slides a non-overlapping `chunk_sec` window.  A chunk is kept if its RMS
    (dBFS) is at least `db_above_median` dB above the per-file median.

    The adaptive threshold means each file self-calibrates to its own noise
    floor — no magic absolute dBFS value needed.

    Parameters
    ----------
    freq_min / freq_max
        Bandpass range in Hz.  Tune per species group:
          frogs/toads   →  100 –  4 000 Hz
          mammals       →   50 –  8 000 Hz  (bats: 10 000 – 120 000)
          insects       →  500 – 20 000 Hz
          broadband     →  100 – 16 000 Hz  ← default; safe starting point
    db_above_median
         3  → aggressive  (catches faint/distant calls, more false positives)
         6  → balanced    ← default
        10  → conservative (only prominent peaks)

    Returns
    -------
    List of (start_sec, end_sec) tuples for chunks that exceed the threshold.
    """
    audio, sr = sf.read(audio_path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)              # stereo → mono

    nyq          = sr / 2.0
    freq_max_clipped = min(freq_max, nyq * 0.95)   # stay below Nyquist

    sos      = scipy_signal.butter(
        4, [freq_min / nyq, freq_max_clipped / nyq], btype="band", output="sos"
    )
    audio_bp = scipy_signal.sosfilt(sos, audio)

    chunk_samples = int(chunk_sec * sr)
    duration      = len(audio) / sr

    energies: list[float]               = []
    bounds:   list[tuple[float, float]] = []

    for chunk_start in range(0, int(duration - chunk_sec + 1), int(chunk_sec)):
        s     = int(chunk_start * sr)
        chunk = audio_bp[s : s + chunk_samples]
        rms   = np.sqrt(np.mean(chunk ** 2))
        energies.append(20.0 * np.log10(rms + 1e-10))
        bounds.append((float(chunk_start), float(chunk_start + chunk_sec)))

    if not energies:
        return []

    threshold = float(np.median(energies)) + db_above_median
    return [(s, e) for (s, e), en in zip(bounds, energies) if en >= threshold]

In [ ]:
# Manifest generator
import soundfile as sf
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
import logging
import contextlib
import io
import torchaudio
import pandas as pd

logging.getLogger("birdnetlib").setLevel(logging.ERROR)

# Initialise once — expensive to reload each call
analyzer = Analyzer()
silent = contextlib.redirect_stdout(io.StringIO())

BIRDNET_SR        = 48000
BIRDNET_CHUNK_SEC = 3
BIRDNET_MIN_CONF  = 0.35   # lower = permissive (catches quiet/distant birds)
                           # raise to 0.25+ if you're getting too much noise through

def generate_manifest(root_dir, output_csv="manifest.csv"):
    manifest_data = []
    skipped_no_bird = 0
    skipped_no_peak = 0

    full_df     = pd.read_csv(os.path.join(root_dir, "train.csv"))

    for _, entry in tqdm(full_df.iterrows(), total=len(full_df)):
        curr_audio_loc = os.path.join(root_dir, "train_audio", entry["filename"])
        label = entry['primary_label']

        try:
            if entry['class_name'] == 'Aves':
                # ONE BirdNET call for the whole file — no temp files, no manual chunking
                rec = Recording(analyzer, curr_audio_loc, min_conf=BIRDNET_MIN_CONF)
                with silent:
                    rec.analyze()

                # Build a lookup of which 3s windows had detections
                # BirdNET uses 1.5s steps internally so detections won't perfectly 
                # align to our chunk boundaries — check for overlap instead
                detections = rec.detections  # list of dicts with start_time, end_time

                # Get file duration without loading the whole waveform
                info     = sf.info(curr_audio_loc)
                duration = info.duration

                for chunk_start in range(0, int(duration - BIRDNET_CHUNK_SEC + 1), BIRDNET_CHUNK_SEC):
                    chunk_end = chunk_start + BIRDNET_CHUNK_SEC

                    if not _window_has_detection(detections, chunk_start, chunk_end):
                        skipped_no_bird += 1
                        continue

                    manifest_data.append({
                        'filename':      curr_audio_loc,
                        'start_sec':     float(chunk_start),
                        'end_sec':       float(chunk_end),
                        'class':         entry['class_name'],
                        'primary_label':    label,
                        'secondary_labels': entry.get('secondary_labels', '[]'),
                    })
            else:
                # ── Non-Aves: spectrogram energy peak detection ──────────────
                active_chunks = _get_spectrogram_active_chunks(curr_audio_loc)

                if not active_chunks:
                    skipped_no_peak += 1
                    continue

                for chunk_start, chunk_end in active_chunks:
                    manifest_data.append({
                        'filename':      curr_audio_loc,
                        'start_sec':     chunk_start,
                        'end_sec':       chunk_end,
                        'class':         entry['class_name'],
                        'primary_label':    label,
                        'secondary_labels': entry.get('secondary_labels', '[]'),
                    })

        except Exception as e:
            print(f"Error processing {curr_audio_loc}: {e}")

    df = pd.DataFrame(manifest_data)
    df.to_csv(output_csv, index=False)
    print(f"Manifest created: {len(df)} chunks kept, {skipped_no_bird} rejected.")


def _window_has_detection(detections: list, chunk_start: float, chunk_end: float) -> bool:
    """
    True if any BirdNET detection overlaps with [chunk_start, chunk_end].
    BirdNET's windows are 3s with 1.5s steps, so a detection at t=1.5
    covers [1.5, 4.5] and overlaps both chunk 0-3 and chunk 3-6.
    """
    for det in detections:
        if det['start_time'] < chunk_end and det['end_time'] > chunk_start:
            return True
    return False

# generate_manifest(root_dir=os.path.join("..", "birdclef-2026"), output_csv=os.path.join("..", "birdclef-2026", "train_preproc_non_ave.csv"))

Dataset and CNN

In [ ]:
# Helper - Guarantee at least one entry per class in each split
import warnings

def stratified_split(df, val_size=0.2, label_col='primary_label', random_state=42):
    """
    Train / val split that guarantees every class appears in both sets.

    Unlike sklearn's stratified split, this handles ALL class sizes correctly:
      - 1 sample  -> duplicated into both splits (with a warning)
      - 2 samples -> 1 val, 1 train (guaranteed, sklearn rounds this to 0 val)
      - 3+ samples -> proportional split, always at least 1 in each half

    The sklearn approach was causing the RuntimeError because for classes with
    very few samples (e.g. 2-4), round(n * val_size) evaluates to 0, leaving
    those classes absent from val entirely.

    Parameters
    ----------
    df           : manifest DataFrame (output of generate_manifest)
    val_size     : target fraction for val (default 0.2)
    label_col    : column containing the class label
    random_state : for reproducible shuffling within each class group

    Returns
    -------
    df_train, df_val — DataFrames with reset indices
    """
    train_rows = []
    val_rows   = []
    warn_list  = []

    for label, group in df.groupby(label_col, sort=False):
        n     = len(group)
        group = group.sample(frac=1, random_state=random_state)  # deterministic shuffle

        if n == 1:
            # Only one sample — duplicate into both so the class stays visible at eval
            train_rows.append(group)
            val_rows.append(group)
            warn_list.append(label)
        else:
            # Clamp: always at least 1 in val AND at least 1 in train
            n_val = max(1, min(n - 1, round(n * val_size)))
            val_rows.append(group.iloc[:n_val])
            train_rows.append(group.iloc[n_val:])

    if warn_list:
        warnings.warn(
            f"{len(warn_list)} class(es) have only 1 sample and will be duplicated "
            f"into both splits: {warn_list}",
            UserWarning,
            stacklevel=2,
        )

    df_train = pd.concat(train_rows, ignore_index=True)
    df_val   = pd.concat(val_rows,   ignore_index=True)

    # Sanity check
    missing_from_train = set(df[label_col]) - set(df_train[label_col])
    missing_from_val   = set(df[label_col]) - set(df_val[label_col])
    if missing_from_train or missing_from_val:
        raise RuntimeError(
            f"BUG: missing from train: {missing_from_train} | "
            f"missing from val: {missing_from_val}"
        )

    return df_train, df_val

In [ ]:
# Dataset class
from torch.utils.data import Dataset
import ast

class BirbSet(Dataset):
    # Possibly add a download flag, for now assume it is on device
    # Add a Train flag to retrieve the sets accordingly
    def __init__(self, df, root, clip_length, label_to_idx, is_train = False, sample_rate= 32000):
        self.clips = []
        self.start_times = []
        self.end_times = []
        self.labels = []
        self.secondary_labels = []

        self.sample_rate = sample_rate
        self.clip_length = clip_length
        self.label_to_idx = label_to_idx
        self.is_train = is_train
        self.root = root

        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.mel_spect = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=800,
            n_mels=64
        )

        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=16)
        
        for i, entry in df.iterrows():
            curr_audio_loc = os.path.normpath(entry["filename"])
            self.clips.append(curr_audio_loc)
            self.labels.append(self.label_to_idx[entry['primary_label']])
            self.start_times.append(entry["start_sec"])
            self.end_times.append(entry["end_sec"])

    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        audio_clip = self.clips[idx]
        label      = self.labels[idx]
        try:
            frame_offset = int(self.start_times[idx] * self.sample_rate)
            num_frames   = int((self.end_times[idx] - self.start_times[idx]) * self.sample_rate)

            waveform, sr = torchaudio.load(
                audio_clip, frame_offset=frame_offset, num_frames=num_frames
            )

            if sr != self.sample_rate:
                waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)

            chunk_size  = self.sample_rate * self.clip_length
            current_len = waveform.shape[1]

            if current_len > chunk_size:
                waveform = waveform[:, :chunk_size]
            elif current_len < chunk_size:
                waveform = torch.nn.functional.pad(waveform, (0, chunk_size - current_len))

            spectrogram = self.mel_spect(waveform)
            spectrogram = self.amp_to_db(spectrogram)
            mean, std   = spectrogram.mean(), spectrogram.std() + 1e-6
            spectrogram = (spectrogram - mean) / std

            target = torch.zeros(len(self.label_to_idx), dtype=torch.float32)

            primary = self.labels[idx]
            target[primary] = 1.0

            raw_secondary = self.secondary_labels[idx]
            if raw_secondary and raw_secondary not in ('[]', '', None):
                for sec_label in ast.literal_eval(raw_secondary):   # FIX 2: was iterating over label_to_idx keys instead of parsed list
                    if sec_label in self.label_to_idx:
                        target[self.label_to_idx[sec_label]] = 1.0

            # if self.is_train:
            #     spectrogram = self.freq_mask(spectrogram)
            #     spectrogram = self.time_mask(spectrogram)

            return spectrogram, target
        except Exception as e:
            print(f"Failed to load audio clip at index {idx} -> {e}.\nFile location {self.clips[idx]}")

In [ ]:
# CNN
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

class EfficientBirbNN(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        
        # 1. Load the base EfficientNet model
        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        self.base_model = efficientnet_b3(weights=weights)
        
        # 2. Modify the first convolutional layer to accept 1-channel spectrograms
        # EfficientNet's first layer is located at self.base_model.features[0][0]
        original_conv = self.base_model.features[0][0]
        self.base_model.features[0][0] = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
        
        # (Optional but recommended) Initialize the new 1-channel conv 
        # by averaging the pre-trained 3-channel weights to retain feature extraction strength
        if pretrained:
            with torch.no_grad():
                self.base_model.features[0][0].weight[:] = original_conv.weight.mean(dim=1, keepdim=True)
                
        # 3. Modify the final classification layer for your specific number of bird classes
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4), # Extra dropout to prevent overfitting on audio data
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

In [ ]:
# Generate data sets
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

root_path         = os.path.join("..", "birdclef-2026")
BIRDNET_CHUNK_SEC = 3.0

# 1. Load the master CSV
full_df = pd.read_csv(os.path.join(root_path, "train_preproc.csv"))

# 2. Stratified split — guarantees every class appears in both halves
#    Replaces the previous file-based split which could leave classes
#    entirely absent from val when a species has only one source file.
df_train, df_val = stratified_split(full_df, val_size=0.2, label_col='primary_label')

# 3. Universal label mapping (sorted for reproducibility)
unique_labels       = sorted(full_df['primary_label'].unique())
master_label_to_idx = {label: i for i, label in enumerate(unique_labels)}
num_classes         = len(master_label_to_idx)

# 4. pos_weight for BCEWithLogitsLoss
#    pos_weight[c] = n_negative[c] / n_positive[c]
#    Upweights rare classes so the loss treats them fairly regardless of
#    how underrepresented they are. Clamped to 20 to prevent extreme
#    gradients for species with only 1-2 training samples.
n          = len(df_train)
pos_counts = torch.zeros(num_classes)

for label in df_train['primary_label']:
    pos_counts[master_label_to_idx[label]] += 1

for sec_str in df_train['secondary_labels'].fillna('[]'):
    for sec in ast.literal_eval(sec_str):
        if sec in master_label_to_idx:
            pos_counts[master_label_to_idx[sec]] += 1

neg_counts = n - pos_counts
pos_weight = (neg_counts / pos_counts.clamp(min=1)).clamp(max=20.0)
# pos_weight is moved .to(device) in the training cell below

# 5. DataLoaders
dset_train = BirbSet(
    df=df_train, root=root_path, clip_length=BIRDNET_CHUNK_SEC,
    label_to_idx=master_label_to_idx, is_train=True     # is_train=True enables augmentation
)
loader = DataLoader(dset_train, batch_size=32, shuffle=True, pin_memory=True)

dset_val = BirbSet(
    df=df_val, root=root_path, clip_length=BIRDNET_CHUNK_SEC,
    label_to_idx=master_label_to_idx, is_train=False
)
loader_val = DataLoader(dset_val, batch_size=32, shuffle=False, pin_memory=True)


Training and validation

In [ ]:
# Hyper params
device = torch.device("cuda")
model = EfficientBirbNN(num_classes=len(unique_labels)).to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

In [ ]:
# New Soundscape training
@torch.inference_mode()
def predict_soundscape(
    model,
    audio_path,
    label_to_idx,
    clip_length    = 5,
    sample_rate    = 32000,
    step_sec       = 5,       # non-overlapping for BirdCLEF submission
    threshold      = 0.5,
    device         = 'cuda',
):
    """
    Returns a DataFrame with one row per 5-second chunk:
        columns = ['filename', 'end_time'] + [species_1, species_2, ...]
    matching the BirdCLEF submission format.
    """
    idx_to_label = {v: k for k, v in label_to_idx.items()}
    model.eval().to(device)

    mel_spect  = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate, n_fft=800, n_mels=64
    ).to(device)
    amp_to_db  = torchaudio.transforms.AmplitudeToDB(stype='power').to(device)

    waveform, sr = torchaudio.load(audio_path)
    if sr != sample_rate:
        waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)   # stereo → mono

    chunk_samples = sample_rate * clip_length
    step_samples  = sample_rate * step_sec
    total_samples = waveform.shape[1]

    rows = []
    for start in range(0, total_samples - chunk_samples + 1, step_samples):
        chunk = waveform[:, start : start + chunk_samples].to(device)

        spec  = mel_spect(chunk)
        spec  = amp_to_db(spec)
        spec  = (spec - spec.mean()) / (spec.std() + 1e-6)
        spec  = spec.unsqueeze(0)                    # (1, 1, freq, time)

        logits = model(spec)                         # (1, n_classes)
        probs  = torch.sigmoid(logits).squeeze(0).cpu()

        end_time = (start + chunk_samples) / sample_rate
        row = {'filename': audio_path, 'end_time': end_time}
        for idx, prob in enumerate(probs):
            row[idx_to_label[idx]] = round(prob.item(), 4)
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Train and validate
from sklearn.metrics import f1_score, average_precision_score
import numpy as np
import copy
from tqdm import tqdm


def validate(model, loader_val, loss_fn, master_label_to_idx, device, threshold=0.5):
    """
    Multi-label validation.

    Reports val loss, macro/micro F1, and mAP — the primary BirdCLEF metric.
    Only classes that are actually present in the val set contribute to mAP,
    so the score is not diluted by absent species.
    """
    idx_to_label = {v: k for k, v in master_label_to_idx.items()}
    n_classes    = len(master_label_to_idx)

    model.eval()
    all_probs   = []
    all_targets = []
    val_losses  = []

    with torch.no_grad():
        for spects, targets in tqdm(loader_val, desc="Validating"):
            spects, targets = spects.to(device), targets.to(device)
            logits = model(spects)
            probs  = torch.sigmoid(logits)           # multi-label: sigmoid per class

            val_losses.append(loss_fn(logits, targets).item())
            all_probs.append(probs.cpu())
            all_targets.append(targets.cpu())

    all_probs   = torch.cat(all_probs).numpy()    # (N, n_classes)
    all_targets = torch.cat(all_targets).numpy()  # (N, n_classes)
    preds       = (all_probs >= threshold).astype(int)

    avg_loss = float(np.mean(val_losses))
    macro_f1 = f1_score(all_targets, preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_targets, preds, average='micro', zero_division=0)

    # ── Per-class stats — skip classes absent from this val batch ─────────────
    class_stats = []
    for c in range(n_classes):
        support = int(all_targets[:, c].sum())
        if support == 0:
            continue
        f1_c = f1_score(all_targets[:, c], preds[:, c], zero_division=0)
        ap_c = average_precision_score(all_targets[:, c], all_probs[:, c])
        class_stats.append((idx_to_label[c], f1_c, ap_c, support))

    mAP = float(np.mean([ap for _, _, ap, _ in class_stats])) if class_stats else 0.0

    print(f"\nVal loss: {avg_loss:.4f} | Macro F1: {macro_f1:.4f} | "
          f"Micro F1: {micro_f1:.4f} | mAP: {mAP:.4f}")
    print(f"Evaluated on {len(class_stats)}/{n_classes} classes present in val set")

    # ── Prediction coverage ───────────────────────────────────────────────────
    any_pred    = preds.any(axis=1)
    exact_match = (preds == all_targets.astype(int)).all(axis=1)
    print(f"\nPrediction coverage:")
    print(f"  Samples with >=1 prediction : {any_pred.sum()} / {len(preds)}")
    print(f"  Exact-match accuracy         : {exact_match.mean() * 100:.2f}%")

    # ── Best / worst 15 classes by AP ────────────────────────────────────────
    header  = f"\n{'Class':<30} {'F1':>7} {'AP':>7} {'Support':>9}"
    divider = "─" * 57

    print(f"\n─── 15 worst classes by AP ───{header}\n{divider}")
    for name, f1_c, ap_c, sup in sorted(class_stats, key=lambda x: x[2])[:15]:
        print(f"{name:<30} {f1_c:>7.3f} {ap_c:>7.3f} {sup:>9}")

    print(f"\n─── 15 best classes by AP ───{header}\n{divider}")
    for name, f1_c, ap_c, sup in sorted(class_stats, key=lambda x: x[2], reverse=True)[:15]:
        print(f"{name:<30} {f1_c:>7.3f} {ap_c:>7.3f} {sup:>9}")

    return avg_loss, mAP


# ── Training loop ──────────────────────────────────────────────────────────────
epochs            = 30
best_val_metric   = 0.0
best_model_state  = None
patience          = 8
epochs_no_improve = 0

# pos_weight moves to device here — computed in the data cell above
loss_fn   = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

for epoch in range(epochs):
    # ── Training ───────────────────────────────────────────────────────────────
    model.train()
    train_losses = []
    for spects, targets in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [train]"):
        spects, targets = spects.to(device), targets.to(device)
        optimiser.zero_grad()
        l = loss_fn(model(spects), targets)
        l.backward()
        optimiser.step()
        train_losses.append(l.item())

    scheduler.step()

    # ── Validation ─────────────────────────────────────────────────────────────
    avg_train = float(np.mean(train_losses))
    avg_val, val_metric = validate(
        model, loader_val, loss_fn, master_label_to_idx, device
    )

    print(f"Epoch {epoch+1:2d} | Train loss: {avg_train:.4f} | "
          f"Val loss: {avg_val:.4f} | mAP: {val_metric:.4f}")

    # ── Early stopping (maximise mAP, the BirdCLEF metric) ────────────────────
    if val_metric > best_val_metric:
        best_val_metric   = val_metric
        best_model_state  = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        torch.save(best_model_state, "best_birb_model.pt")
        print(f"  -> New best model saved (mAP={best_val_metric:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_model_state)